In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pickle
import librosa
import torch
import torch.nn as nn

from transformers import (
    HubertModel,
    HubertForCTC,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    AutoTokenizer,
    AutoModel
)

SAVE_DIR = "/content/drive/MyDrive/IIITH_Text"

MODEL_PATH = f"{SAVE_DIR}/multimodal_emotion_model.pt"

LABEL_ENCODER_PATH = f"{SAVE_DIR}/label_encoder.pkl"

TEST_AUDIO = "/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/OAF_happy/OAF_bar_happy.wav"

TEXT_MODEL = "bert-base-uncased"

HUBERT_ENCODER = "facebook/hubert-base-ls960"

HUBERT_ASR = "facebook/hubert-large-ls960-ft"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", DEVICE)


with open(LABEL_ENCODER_PATH, "rb") as f:

    label_encoder = pickle.load(f)

NUM_CLASSES = len(label_encoder.classes_)


text_tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL
)


print("\nLoading HuBERT Models...")

feature_extractor = (
    Wav2Vec2FeatureExtractor
    .from_pretrained(HUBERT_ENCODER)
)

hubert_encoder = HubertModel.from_pretrained(
    HUBERT_ENCODER,
    output_hidden_states=True
).to(DEVICE)

processor = Wav2Vec2Processor.from_pretrained(
    HUBERT_ASR
)

hubert_ctc = HubertForCTC.from_pretrained(
    HUBERT_ASR
).to(DEVICE)

hubert_encoder.eval()
hubert_ctc.eval()

print("HuBERT Models Loaded!")


class AttentionPooling(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.attention = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, x):

        weights = torch.softmax(
            self.attention(x),
            dim=1
        )

        pooled = torch.sum(
            weights * x,
            dim=1
        )

        return pooled

# EXACT TRAINING MODEL

class MultimodalEmotionModel(nn.Module):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()


        self.speech_lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        self.speech_attention = AttentionPooling(
            512
        )

        self.speech_fc = nn.Linear(
            512,
            256
        )


        self.text_encoder = AutoModel.from_pretrained(
            TEXT_MODEL
        )

        self.text_fc = nn.Linear(
            768,
            256
        )


        self.cross_attention = nn.MultiheadAttention(
            embed_dim=256,
            num_heads=4,
            batch_first=True
        )


        self.classifier = nn.Sequential(

            nn.Linear(512, 256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        speech,
        input_ids,
        attention_mask
    ):


        speech_out, _ = self.speech_lstm(
            speech
        )

        speech_repr = self.speech_attention(
            speech_out
        )

        speech_repr = self.speech_fc(
            speech_repr
        )


        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_repr = text_outputs.last_hidden_state[:,0]

        text_repr = self.text_fc(
            text_repr
        )


        speech_repr = speech_repr * 0.7

        text_repr = text_repr * 0.3


        speech_q = speech_repr.unsqueeze(1)

        text_kv = text_repr.unsqueeze(1)

        attended, _ = self.cross_attention(
            query=speech_q,
            key=text_kv,
            value=text_kv
        )

        attended = attended.squeeze(1)

        

        fused = torch.cat([
            speech_repr,
            attended
        ], dim=1)

        logits = self.classifier(
            fused
        )

        return logits


print("\nLoading Multimodal Model...")

model = MultimodalEmotionModel(
    num_classes=NUM_CLASSES
).to(DEVICE)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

if "model_state_dict" in checkpoint:

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

else:

    model.load_state_dict(
        checkpoint
    )

model.eval()

print("Model Loaded Successfully!")


def transcribe_audio(audio_path):

    waveform, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    inputs = processor(
        waveform,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = (
        inputs.input_values
        .to(DEVICE)
    )

    with torch.no_grad():

        logits = hubert_ctc(
            input_values
        ).logits

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

    transcription = processor.batch_decode(
        predicted_ids
    )[0]

    return transcription.lower()


def extract_hubert_sequence(audio_path):

    waveform, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    waveform, _ = librosa.effects.trim(
        waveform,
        top_db=20
    )

    inputs = feature_extractor(
        waveform,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )

    input_values = (
        inputs["input_values"]
        .to(DEVICE)
    )

    with torch.no_grad():

        outputs = hubert_encoder(
            input_values
        )

        hidden_states = outputs.hidden_states


    selected_layers = hidden_states[6:13]

    stacked_layers = torch.stack(
        selected_layers,
        dim=0
    )

    weighted_sum = stacked_layers.mean(
        dim=0
    )

    embedding = weighted_sum.squeeze(0)

    return embedding


def encode_text(text):

    encoding = text_tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=32,
        return_tensors="pt"
    )

    input_ids = encoding[
        "input_ids"
    ].to(DEVICE)

    attention_mask = encoding[
        "attention_mask"
    ].to(DEVICE)

    return input_ids, attention_mask


def predict_emotion(audio_path):

    print("\n========================================")

    print("Processing:", audio_path)


    transcription = transcribe_audio(
        audio_path
    )

    print("\nTranscript:")
    print(transcription)


    speech_features = extract_hubert_sequence(
        audio_path
    )

    speech_features = (
        speech_features
        .unsqueeze(0)
        .to(DEVICE)
    )


    input_ids, attention_mask = encode_text(
        transcription
    )


    with torch.no_grad():

        outputs = model(
            speech_features,
            input_ids,
            attention_mask
        )

        probs = torch.softmax(
            outputs,
            dim=1
        )

        pred = torch.argmax(
            probs,
            dim=1
        ).item()

    predicted_emotion = (
        label_encoder.classes_[pred]
    )

    confidence = (
        probs[0][pred]
        .item()
    )


    print("\nPredicted Emotion:")
    print(predicted_emotion)

    print(
        "\nConfidence:",
        round(confidence * 100, 2),
        "%"
    )

    print("\nAll Emotion Scores:\n")

    for emotion, score in zip(
        label_encoder.classes_,
        probs[0]
    ):

        print(
            f"{emotion:20s}: {score.item():.4f}"
        )

    print("\n========================================")

# TEST


predict_emotion(TEST_AUDIO)

Using Device: cpu

Loading HuBERT Models...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

HuBERT Models Loaded!

Loading Multimodal Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded Successfully!

Processing: /content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/OAF_happy/OAF_bar_happy.wav

Transcript:
say the word bar

Predicted Emotion:
happy

Confidence: 100.0 %

All Emotion Scores:

angry               : 0.0000
disgust             : 0.0000
fear                : 0.0000
happy               : 1.0000
neutral             : 0.0000
ps                  : 0.0000
sad                 : 0.0000

